In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Lecture 19: A/B Testing ##

The table `baby.csv` contains data on a random sample of 1,174 mothers and their newborn babies. The column Birth Weight contains the birth weight of the baby, in ounces; Gestational Days is the number of gestational days, that is, the number of days the baby was in the womb. There is also data on maternal age, maternal height, maternal pregnancy weight, and whether or not the mother was a smoker.

In [ ]:
births = Table.read_table('baby.csv')

In [ ]:
births.show(2)

In this notebook, we're focused on a possible association between the baby's birth weight and whether or not the mother smoked while pregnant. We make a new table with just those two columns:

In [ ]:
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
# Count the numbers of smokers and non-smokers
smoking_and_birthweight.group('Maternal Smoker')

In [ ]:
# Find the average birthweight (ounces) for babies in each group
smoking_and_birthweight.group('Maternal Smoker', collect=np.average)

In [ ]:
# Look at the distribution of baby birth weights in an overlaid histogram
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

It seems that, in our sample data, there is some association between maternal smoking and baby birth weight. Is it enough of a difference to be statistically significant, indicating an actual difference in the underlying population? (Is maternal smoked associated with lower baby birthweight in the general population?)

**Hypotheses**:
  - The null hypothesis states that the observed difference is only due to random variation.
  - The alternative states that, in the general population of mothers and babies, maternal smoking is associated with lower baby birth weights.

**Test Statistic**:
Let A be the group where the mothers did not smoke, and let B be the group where the mothers smoked. Our test statistic will be the difference in average birthweight between the two groups, specifically, `mean_birthweight_B - mean_birthweight_A`. Notice that **more negative** values of the statistic support the alternative hypothesis.

In [ ]:
# Assign a name to the table showing the average birthweight for each group:
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

### Find the observed test statistic ###

In [ ]:
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

### Write a Helper Function to Calculate the Test Statistic ###

In [ ]:
def difference_of_means(table, num_label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(num_label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)

    means = means_table.column(1)
    return means.item(1) - means.item(0)

In [ ]:
# The following call should give the same result we found above, the
# observed value of our test statistic in the sample data
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

### Shuffles: Simulation Under Null Hypothesis ###

If the null hypothesis is true, then randomly shuffling the True/False values in the maternal smoking column should have a minimal effect on the test statistic.

In [ ]:
# Recall our data
smoking_and_birthweight.show(4)

In [ ]:
# This is how we can use tbl.sample() to shuffle a table...
smoking_and_birthweight.sample(with_replacement=False)

In [ ]:
# After shuffling the table, extract the shuffled labels
shuffled_labels = (smoking_and_birthweight
    .sample(with_replacement=False)
    .column('Maternal Smoker')
)
shuffled_labels[0:10]

In [ ]:
# Make a new table by adding a third column (shuffled labels) to our
# original table
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)

In [ ]:
original_and_shuffled

In [ ]:
# Genereate one simulated value of our test statistic under the null
difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
# Generate the observed test statistic (actual difference of means in the data)
difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')

### Permutation Test ###

To carry out our test, we need to simulate thousands of test statistics, under the assumption that the null hypothesis is true. We do this by repeatedly shuffling the Maternal Smoking values and calculating the difference in average birthweights between the two groups. 

Write a function to generate one simulated test statistic:

In [ ]:
def one_simulated_difference(table, num_label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False
                                                    ).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(num_label).with_column(
        'Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, num_label, 'Shuffled Label')   

In [ ]:
# a simulated difference -- check that it varies randomly when we re-run the cell
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# Now that we can simulate one difference, we need to generate an array of thousands
# of differences for our histogram. This will take 10-20 seconds, permutation testing
# is computationally demanding.

differences = make_array()

for i in np.arange(2500):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

print(len(differences))
differences[0:10]

In [ ]:
# Now for the histogram:
Table().with_column('Difference Between Group Means', differences).hist()
plots.scatter(observed_difference, -0.01, color='red', s=50);
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');


### Find the P-Value ###

The P-value is the area in the histogram in the direction of support for the alternative hypothesis. Here, more-negative values support the alternative, so P = left-tail area. 

None of the simulated differences are left of the red dot, so P = 0. This is the strongest-possible P-value. 

### Reach a conclusion ###

We have highly significant evidence that the alternative hypothesis is true: in the general population, babies of mothers who smoke tend to weigh less than do babies of mothers who do not smoke.